# PII Detector Comparison

Benchmark **9 PII detectors** — Skyflow Detect, OPF, default GLiNER + 3 GLiNER variants (Nvidia, Gretel small/large), Microsoft Presidio, the ai4privacy ModernBERT model, and the OpenMed PII family — against a sample from any of 5 ai4privacy datasets. Same harness as `eval/src/opf_eval/` — exposed via a notebook for easy hosted runs.

**What this notebook does:**
1. Installs the comparison harness + open-weight detectors (OPF, GLiNER family, ai4privacy, OpenMed) and Presidio + spaCy models
2. Materializes a deterministic fixture set from your chosen dataset
3. Runs each detector you select against the fixtures, writing `raw_<detector>.jsonl` per detector
4. Renders a Markdown report (fair view + raw view + per-language SemEval Type F1)
5. Optional: bar charts of per-category F1, latency comparison

**You'll need:**
- For the local detectors (OPF / GLiNER family / Presidio / ai4privacy / OpenMed): Colab free tier is enough (CPU works; GPU optional)
- For Skyflow Detect: a vault URL, vault ID, and bearer token (set as Colab secrets)

**Estimated wall time** at default config (100 examples, 3 local detectors): ~5 minutes. Adding all 9 detectors at 1k examples is closer to ~40 minutes (OpenMed loads a separate model per language; Nvidia + Gretel large download ~2 GB each).

## 1. Setup — install the harness and dependencies

Run this once per Colab session. Installs:
- The OPF source repo (open-weight PII detector from OpenAI)
- The comparison harness (this repo's `eval/` package)
- GLiNER, Presidio, and supporting libs

**Edit `HARNESS_REPO` below** to point at the fork/branch where this notebook's source lives.

In [ ]:
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
OPF_REPO = "https://github.com/openai/privacy-filter.git"

import os, subprocess, sys

# Triton has no stable Apple Silicon support and isn't needed on Colab CPU/GPU.
# Setting before any opf import keeps the runtime on the vanilla PyTorch MoE path.
os.environ.setdefault("OPF_MOE_TRITON", "0")

PIP = f"{sys.executable} -m pip"  # ensure we install into the notebook's kernel

def _run(cmd, *, msg, show_output=False):
    print(f"==> {msg}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 or show_output:
        if r.stdout: print(r.stdout)
        if r.stderr: print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{msg} failed (exit {r.returncode}). See output above.")

if not os.path.exists("/content/privacy-filter"):
    _run(f"git clone --depth 1 {OPF_REPO} /content/privacy-filter", msg="clone privacy-filter")
if not os.path.exists("/content/local-privacy"):
    _run(f"git clone --depth 1 {HARNESS_REPO} /content/local-privacy", msg="clone local-privacy (must be public, or use a PAT in HARNESS_REPO)")

# Pip-install for dependency resolution (torch, datasets, gliner, presidio, etc).
# Whether or not pip places the top-level packages in site-packages reliably on
# Colab, we also add the source directories to sys.path below so imports always
# work.
_run(f"{PIP} install -q /content/privacy-filter", msg="install opf + deps")
_run(f"{PIP} install -q /content/local-privacy/eval", msg="install opf-eval + deps")

# Belt-and-suspenders: expose the source trees on sys.path so `import opf` and
# `import opf_eval` resolve regardless of what pip did with the editable hooks.
for src_dir in ("/content/privacy-filter", "/content/local-privacy/eval/src"):
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

# Confirm the kernel can import everything we need.
import importlib
for mod in ("opf", "opf_eval", "opf_eval.runner", "opf_eval.report", "opf_eval.fixtures"):
    importlib.import_module(mod)
    print(f"  ok: {mod}")

print("\nsetup complete.")


In [ ]:
# spaCy models for Presidio. en_core_web_lg is required; the others are
# optional and only needed if you want multilingual Presidio (skip if running
# default English-only Presidio — most cases).
!python -m spacy download en_core_web_lg -q

# Uncomment the next 5 lines for multilingual Presidio (~3 GB extra download):
# !python -m spacy download nl_core_news_lg -q
# !python -m spacy download fr_core_news_lg -q
# !python -m spacy download de_core_news_lg -q
# !python -m spacy download it_core_news_lg -q
# !python -m spacy download es_core_news_lg -q

print("\nspaCy models ready.")

## 2. Imports and configuration

Tweak the config cell to change sample size, dataset, detectors, and output location.

**Datasets:**
- `pii_masking_300k` (default), `pii_masking_200k`, `pii_masking_400k` — legacy ai4privacy variants (different vocabularies)
- `openpii_nano` (1k), `openpii_mini` (10k) — OpenPII vocabulary

**Detector names:**
- `opf` — OpenAI Privacy Filter (open-weight, local) — overall local leader
- `gliner` — default GLiNER multilingual PII (`urchade/gliner_multi_pii-v1`) — prompts auto-restricted to dataset vocab
- `gliner_nvidia` — Nvidia gliner-PII on `urchade/gliner_large-v2.1` (570M base, threshold 0.3, NVIDIA Open Model License) — strongest of the GLiNER variants
- `gliner_gretel_small` / `gliner_gretel_large` — Gretel bi-encoder GLiNER (threshold 0.7, English-only training, snake_case 41-label vocab)
- `ai4privacy_modernbert` — ai4privacy ModernBERT-base (~150M, MIT, 8 languages, OpenPII vocab)
- `openmed` — OpenMed PII via `openmed.extract_pii(lang=…)` — DeBERTa-based per-language models, snake_case 55-label vocab
- `presidio` — Microsoft Presidio English-only (regex + NER, local)
- `presidio_multilang` — Presidio with all 6 spaCy models (requires the optional downloads above)
- `skyflow` — Skyflow Detect API; `entity_types` auto-derived from dataset canonicals (requires creds)
- `skyflow_full` — Skyflow Detect API with all ~70 entity types (requires creds)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

from opf_eval import fixtures, runner, report
from opf_eval.datasets import DEFAULT_DATASET

# === EDIT THESE ===
DATASET = DEFAULT_DATASET     # one of: pii_masking_300k, pii_masking_200k, pii_masking_400k, openpii_nano, openpii_mini
N_EXAMPLES = 100              # 100 for a quick smoke test, 1000 for a real bench, 5000 for stable signal
DETECTORS = ["presidio", "gliner", "opf"]   # add "skyflow" after setting Skyflow creds below
FIXTURE_SEED = 42             # deterministic sample
RUN_NAME = "colab_demo"       # used as the output dir
# ===================

FIXTURES_PATH = Path(f"/content/data/{DATASET}_{N_EXAMPLES}.jsonl")
OUT_DIR = Path(f"/content/results/{RUN_NAME}")

FIXTURES_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"dataset:                {DATASET}")
print(f"will write fixtures to: {FIXTURES_PATH}")
print(f"will write results to:  {OUT_DIR}")
print(f"detectors:              {DETECTORS}")

## 3. Skyflow credentials (optional — skip if not benchmarking Skyflow)

Add these to **Colab Secrets** (key icon in the left sidebar):
- `SKYFLOW_VAULT_URL` — e.g. `https://abc123.vault.skyflowapis.com`
- `SKYFLOW_VAULT_ID` — vault UUID
- `SKYFLOW_BEARER_TOKEN` — short-lived bearer token

Then run the cell below to expose them as env vars.

In [ ]:
try:
    from google.colab import userdata
    for var in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN"):
        try:
            os.environ[var] = userdata.get(var)
        except Exception:
            print(f"  {var}: not set in Colab Secrets")
    if all(v in os.environ for v in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN")):
        print("Skyflow creds loaded.")
    else:
        print("Skyflow creds incomplete — skip the skyflow detectors in DETECTORS.")
except ImportError:
    print("Not in Colab. Set SKYFLOW_VAULT_URL/SKYFLOW_VAULT_ID/SKYFLOW_BEARER_TOKEN in your shell env instead.")

## 4. Materialize the fixture set

Pulls a deterministic sample from PII-Masking-300k via HuggingFace Datasets, projects gold spans through the canonical taxonomy, writes JSONL.

First run downloads ~700 MB of dataset shards; cached for subsequent runs.

In [ ]:
if not FIXTURES_PATH.exists():
    n = fixtures.materialize(FIXTURES_PATH, N_EXAMPLES, dataset=DATASET, seed=FIXTURE_SEED)
    print(f"wrote {n} examples to {FIXTURES_PATH}")
else:
    print(f"reusing existing fixtures at {FIXTURES_PATH}")

import json
with FIXTURES_PATH.open() as f:
    sample = json.loads(f.readline())
print(f"\nfirst fixture keys: {list(sample.keys())}")
print(f"first fixture preview: {sample['text'][:120]}...")
print(f"first fixture gold spans: {len(sample['gold_spans'])} spans")

## 5. Run the detectors

Each detector's predictions are streamed to `raw_<detector>.jsonl` in `OUT_DIR`. Re-running is idempotent for already-completed detectors thanks to the manifest merge logic.

**Per-detector wall time** at 100 examples (rough, CPU):
- Presidio: ~30s (fastest)
- ai4privacy_modernbert: ~30s (small ModernBERT base)
- gliner_gretel_small: ~45s
- GLiNER (default): ~1 min (first run downloads ~500 MB model)
- gliner_nvidia / gliner_gretel_large: ~3-4 min (570M / 500M models)
- OPF: ~1 min on short inputs, ~10 min at 1k (first run downloads ~2.8 GB checkpoint)
- OpenMed: ~1 min English, +30-60s per additional language as it loads per-language models
- Skyflow: ~2 min (network-bound at 1 req/sec)

In [ ]:
runner.run(
    fixtures=FIXTURES_PATH,
    detector_names=DETECTORS,
    out_dir=OUT_DIR,
    dataset=DATASET,             # threads dataset_canonicals into skyflow + gliner builders
    skyflow_workers=1,           # serial; bump if your Skyflow plan allows
    skyflow_min_interval_ms=0,   # add throttling if rate-limited
)

print("\nfiles written:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}")

## 6. Generate and display the report

Builds the markdown report (headline + per-category + per-language) and renders it inline.

In [ ]:
md = report.build_report(OUT_DIR, FIXTURES_PATH)
(OUT_DIR / "report.md").write_text(md)

display(Markdown(md))

## 7. Optional — visualizations

Bar charts of per-category F1 and a latency comparison. Useful for slide decks.

In [ ]:
import json, statistics
import matplotlib.pyplot as plt
import numpy as np

from opf_eval.metrics import score
from opf_eval.datasets import get as get_dataset_config
from opf_eval.taxonomy import dataset_canonicals

fixture_records = [json.loads(l) for l in FIXTURES_PATH.open() if l.strip()]
fixture_index = {r["id"]: r for r in fixture_records}

def detector_data(name):
    path = OUT_DIR / f"raw_{name}.jsonl"
    if not path.exists():
        return None
    records = [json.loads(l) for l in path.open() if l.strip()]
    pairs = []
    latencies = []
    for r in records:
        if r.get("error"):
            continue
        gold = fixture_index[r["id"]]["gold_spans"]
        pairs.append((r["spans"], gold))
        latencies.append(r["latency_ms"])
    return score(pairs), latencies

# Auto-discover every detector with a raw_*.jsonl in OUT_DIR. Picks up
# detectors added by later cells without needing to edit the DETECTORS
# list at the top.
all_detectors = sorted(p.stem.removeprefix("raw_") for p in OUT_DIR.glob("raw_*.jsonl"))
results = {d: detector_data(d) for d in all_detectors if detector_data(d)}
print(f"detectors in this run: {list(results)}")

# Use the chosen dataset's annotated canonicals as the chart x-axis.
# Dataset-specific so we show MONEY/OCCUPATION/ORGANIZATION/VEHICLE/PHYSICAL
# bars only when the dataset actually annotates them.
vocab_key = get_dataset_config(DATASET).vocab_key
labels = sorted(dataset_canonicals(vocab_key))
print(f"chart labels: {labels}")

# === Per-category F1 bar chart ===
x = np.arange(len(labels))
bar_w = 0.8 / max(len(results), 1)
fig, ax = plt.subplots(figsize=(max(12, 0.9 * len(labels)), 5))
for i, (det, (rep, _)) in enumerate(results.items()):
    f1s = [rep.per_label_partial.get(lbl).f1 if rep.per_label_partial.get(lbl) else 0 for lbl in labels]
    ax.bar(x + i * bar_w, f1s, bar_w, label=det)
ax.set_xticks(x + bar_w * (len(results) - 1) / 2)
ax.set_xticklabels(labels, rotation=20)
ax.set_ylabel("F1 (partial overlap, IoU>=0.5)")
ax.set_title(f"Per-category F1 by detector ({DATASET})")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", ncol=2 if len(results) > 4 else 1, fontsize=8)
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# === Latency comparison ===
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
p50s = [statistics.median(results[n][1]) for n in names]
p95s = [sorted(results[n][1])[int(0.95 * (len(results[n][1]) - 1))] for n in names]
p99s = [sorted(results[n][1])[int(0.99 * (len(results[n][1]) - 1))] for n in names]
x = np.arange(len(names))
ax.bar(x - 0.25, p50s, 0.25, label="p50")
ax.bar(x, p95s, 0.25, label="p95")
ax.bar(x + 0.25, p99s, 0.25, label="p99")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("latency (ms)")
ax.set_title("Per-detector latency")
ax.set_yscale("log")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5, which="both")
plt.tight_layout()
plt.show()

## 8. Adding more detectors after the fact

Already ran OPF + GLiNER + Presidio and now want to add Skyflow without re-running the others? Set Skyflow creds (cell 3), then run the cell below. Re-run cell 15 afterward to refresh the charts — it auto-discovers any new `raw_<detector>.jsonl` in `OUT_DIR`.


In [ ]:
runner.run(
    fixtures=FIXTURES_PATH,
    detector_names=["skyflow"],   # only the new one
    out_dir=OUT_DIR,                # same dir
    dataset=DATASET,
)
md = report.build_report(OUT_DIR, FIXTURES_PATH)
(OUT_DIR / "report.md").write_text(md)
display(Markdown(md))


## 9. Saving the run

Colab storage is ephemeral. To keep results, mount Drive and copy the run dir.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
dest = "/content/drive/MyDrive/pii_benchmark_runs/" + OUT_DIR.name
shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
print(f"copied {OUT_DIR} -> {dest}")
